# Notebook 01 — Source Inventory and Data Profiling

**Project:** Enterprise E-Commerce Operations and Customer Experience Control Tower  
**Dataset:** Brazilian E-Commerce Public Dataset by Olist  
**Phase:** 3 — Data Profiling  

## Objectives
- Profile every source CSV file (shape, dtypes, nulls, duplicates, distributions)
- Test uniqueness of all candidate primary keys
- Detect timestamp sequence errors in orders
- Validate numeric ranges (price, freight, review score)
- Validate geographic coordinates
- Produce a data quality report saved to CSV

## Rules
- Raw files are NEVER modified in this notebook
- Every finding is recorded in the data quality report
- No joins happen in this notebook — we only profile individual tables

---
## 0. Setup — Imports and Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import os

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 60)
pd.set_option('display.float_format', '{:,.2f}'.format)

RAW_PATH = '../data/raw/'
OUT_PATH = '../data/processed/'
os.makedirs(OUT_PATH, exist_ok=True)

print('Libraries loaded.')
print('Raw path:', RAW_PATH)
print('Output path:', OUT_PATH)

---
## 1. Load All Source Files

In [ ]:
orders    = pd.read_csv(RAW_PATH + 'olist_orders_dataset.csv')
items     = pd.read_csv(RAW_PATH + 'olist_order_items_dataset.csv')
customers = pd.read_csv(RAW_PATH + 'olist_customers_dataset.csv')
products  = pd.read_csv(RAW_PATH + 'olist_products_dataset.csv')
sellers   = pd.read_csv(RAW_PATH + 'olist_sellers_dataset.csv')
payments  = pd.read_csv(RAW_PATH + 'olist_order_payments_dataset.csv')
reviews   = pd.read_csv(RAW_PATH + 'olist_order_reviews_dataset.csv')
geo       = pd.read_csv(RAW_PATH + 'olist_geolocation_dataset.csv')
cat_trans = pd.read_csv(RAW_PATH + 'product_category_name_translation.csv')

# Parse all datetime columns safely
date_cols = ['order_purchase_timestamp','order_approved_at',
             'order_delivered_carrier_date','order_delivered_customer_date',
             'order_estimated_delivery_date']
for c in date_cols:
    orders[c] = pd.to_datetime(orders[c], errors='coerce')

reviews['review_creation_date']    = pd.to_datetime(reviews['review_creation_date'], errors='coerce')
reviews['review_answer_timestamp'] = pd.to_datetime(reviews['review_answer_timestamp'], errors='coerce')

print('All 9 files loaded successfully.')
datasets = {
    'orders': orders, 'order_items': items, 'customers': customers,
    'products': products, 'sellers': sellers, 'payments': payments,
    'reviews': reviews, 'geolocation': geo, 'category_translation': cat_trans
}
for name, df in datasets.items():
    print(f'  {name:25s}: {df.shape[0]:>8,} rows x {df.shape[1]} cols')

---
## 2. Orders Table Profiling

**Grain:** One row per order  
**Candidate Key:** `order_id`  
**Planned Target:** `fact_orders`

In [ ]:
print('--- Shape and Types ---')
print(orders.shape)
print(orders.dtypes)

In [ ]:
print('--- Null Counts ---')
orders.isnull().sum()

In [ ]:
print('--- Key Uniqueness ---')
print(f'Total rows      : {len(orders):,}')
print(f'Unique order_id : {orders["order_id"].nunique():,}')
print(f'Duplicate keys  : {orders.duplicated("order_id").sum()}')

In [ ]:
print('--- Order Status Distribution ---')
status_dist = orders['order_status'].value_counts().reset_index()
status_dist.columns = ['status', 'count']
status_dist['pct'] = (status_dist['count'] / len(orders) * 100).round(2)
status_dist

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
status_dist.plot(kind='bar', x='status', y='count', ax=ax, color='steelblue', legend=False)
ax.set_title('Order Status Distribution', fontsize=14)
ax.set_xlabel('Status')
ax.set_ylabel('Count')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
print('--- Date Ranges ---')
print(f'Purchase date range : {orders["order_purchase_timestamp"].min()} to {orders["order_purchase_timestamp"].max()}')
print()
print('--- Timestamp Sequence Checks ---')
delivered = orders[orders['order_status'] == 'delivered'].copy()
ts_err      = (delivered['order_delivered_customer_date'] < delivered['order_purchase_timestamp']).sum()
est_err     = (orders['order_estimated_delivery_date'] < orders['order_purchase_timestamp']).sum()
del_mismatch = ((orders['order_status'] == 'delivered') & orders['order_delivered_customer_date'].isnull()).sum()
print(f'Delivery before purchase (sequence error) : {ts_err}')
print(f'Estimated delivery before purchase        : {est_err}')
print(f'Status=delivered but no delivery date     : {del_mismatch}')

---
## 3. Order Items Table Profiling

**Grain:** One row per order item  
**Candidate Key:** `order_id` + `order_item_id` (composite)  
**Planned Target:** `fact_order_items`

In [ ]:
print(items.info())

In [ ]:
print('--- Key and Value Checks ---')
print(f'Rows                             : {len(items):,}')
print(f'Duplicate composite key          : {items.duplicated(["order_id","order_item_id"]).sum()}')
print(f'Missing product_id               : {items["product_id"].isnull().sum()}')
print(f'Missing seller_id                : {items["seller_id"].isnull().sum()}')
print(f'Price <= 0                       : {(items["price"] <= 0).sum()}')
print(f'Freight < 0                      : {(items["freight_value"] < 0).sum()}')
items_per_order = items.groupby('order_id').size()
print(f'Max items in one order           : {items_per_order.max()}')
print(f'Average items per order          : {items_per_order.mean():.2f}')
print(f'GMV (sum of price)               : R$ {items["price"].sum():,.2f}')
print(f'Total freight                    : R$ {items["freight_value"].sum():,.2f}')

In [ ]:
print('--- Price Distribution ---')
items['price'].describe()

---
## 4. Customers Table Profiling

**Grain:** One row per order-customer ID  
**Candidate Key:** `customer_id`  
**Planned Target:** `dim_customers`

In [ ]:
print(f'customer_id unique       : {customers["customer_id"].nunique():,} / {len(customers):,}')
print(f'customer_unique_id unique: {customers["customer_unique_id"].nunique():,}')
print(f'Repeat customers         : {len(customers) - customers["customer_unique_id"].nunique():,}')
print()
print('Top 5 customer states:')
customers['customer_state'].value_counts().head(5)

---
## 5. Products Table Profiling

**Grain:** One row per product  
**Candidate Key:** `product_id`  
**Planned Target:** `dim_products`

In [ ]:
print('--- Null Counts ---')
products.isnull().sum()

In [ ]:
trans_cats = set(cat_trans['product_category_name'].tolist())
prod_cats  = set(products['product_category_name'].dropna().tolist())
untranslated = prod_cats - trans_cats
print(f'Categories without English translation: {len(untranslated)}')
print(f'Categories: {untranslated}')

---
## 6. Sellers Table Profiling

**Grain:** One row per seller  
**Candidate Key:** `seller_id`  
**Planned Target:** `dim_sellers`

In [ ]:
print(f'seller_id unique : {sellers["seller_id"].nunique():,} / {len(sellers):,}')
print(f'Nulls            : {sellers.isnull().sum().sum()}')
print()
print('Sellers by state:')
sellers['seller_state'].value_counts()

---
## 7. Payments Table Profiling

**Grain:** One row per payment sequence per order  
**Candidate Key:** `order_id` + `payment_sequential`  
**Planned Target:** `fact_payments` + aggregated into `fact_orders`

> **CRITICAL:** 2,961 orders have more than one payment row. Joining this directly to orders without aggregation will multiply rows and corrupt all metrics.

In [ ]:
print(f'Total payment rows                : {len(payments):,}')
print(f'Unique order_id in payments       : {payments["order_id"].nunique():,}')
print(f'Duplicate composite key           : {payments.duplicated(["order_id","payment_sequential"]).sum()}')
print(f'Negative payment_value            : {(payments["payment_value"] < 0).sum()}')
print(f'Zero payment_value                : {(payments["payment_value"] == 0).sum()}')
multi_pay = payments.groupby('order_id').size()
print(f'Orders with multiple payment rows : {(multi_pay > 1).sum():,}')
print(f'Max payment rows for one order    : {multi_pay.max()}')

In [ ]:
print('Payment type distribution:')
pay_dist = payments['payment_type'].value_counts().reset_index()
pay_dist.columns = ['payment_type','count']
pay_dist['pct'] = (pay_dist['count'] / len(payments) * 100).round(2)
pay_dist

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
pay_dist.plot(kind='pie', y='count', labels=pay_dist['payment_type'], ax=axes[0],
              autopct='%1.1f%%', legend=False)
axes[0].set_title('Payment Type Share')
payments['payment_installments'].value_counts().sort_index().head(12).plot(
    kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Installment Distribution')
axes[1].set_xlabel('Installments')
plt.tight_layout()
plt.show()

---
## 8. Reviews Table Profiling

**Grain:** One review record  
**Candidate Key:** `review_id` (NOT fully unique — 814 duplicate review_ids found)  
**Planned Target:** `fact_reviews` + aggregated into `fact_orders`

> **CRITICAL:** 547 orders have more than one review row. Must aggregate to order level before joining.

In [ ]:
print(f'Total review rows             : {len(reviews):,}')
print(f'Unique review_id              : {reviews["review_id"].nunique():,}')
print(f'Duplicate review_id           : {reviews.duplicated("review_id").sum()}')
print(f'Unique order_id in reviews    : {reviews["order_id"].nunique():,}')
multi_rev = reviews.groupby('order_id').size()
print(f'Orders with multiple reviews  : {(multi_rev > 1).sum()}')
print()
print('Null counts:')
print(reviews.isnull().sum())

In [ ]:
print('Review score distribution:')
score_dist = reviews['review_score'].value_counts().sort_index().reset_index()
score_dist.columns = ['score','count']
score_dist['group'] = score_dist['score'].map({1:'Negative',2:'Negative',3:'Neutral',4:'Positive',5:'Positive'})
score_dist['pct'] = (score_dist['count'] / len(reviews) * 100).round(1)
score_dist

In [ ]:
colors = ['#e74c3c','#e67e22','#f1c40f','#2ecc71','#27ae60']
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(score_dist['score'], score_dist['count'], color=colors)
axes[0].set_title('Review Score Distribution (1-5)')
axes[0].set_xlabel('Review Score')
axes[0].set_ylabel('Count')

group_summary = score_dist.groupby('group')['count'].sum().reindex(['Negative','Neutral','Positive'])
group_summary.plot(kind='pie', autopct='%1.1f%%', colors=['#e74c3c','#f1c40f','#27ae60'], ax=axes[1])
axes[1].set_title('Review Group Summary')
plt.tight_layout()
plt.show()

---
## 9. Geolocation Table Profiling

**Grain:** Multiple rows per ZIP prefix  
**Candidate Key:** None in raw form  
**Planned Target:** `dim_customer_geography`, `dim_seller_geography`

> **CRITICAL:** 1,000,163 rows for only 19,015 unique ZIP prefixes. Must aggregate to 1 row per ZIP using median lat/lng.

In [ ]:
print(f'Total rows              : {len(geo):,}')
print(f'Unique ZIP prefixes     : {geo["geolocation_zip_code_prefix"].nunique():,}')
print(f'Full row duplicates     : {geo.duplicated().sum():,}')
invalid_lat = ((geo['geolocation_lat'] < -35) | (geo['geolocation_lat'] > 5)).sum()
invalid_lng = ((geo['geolocation_lng'] < -74) | (geo['geolocation_lng'] > -28)).sum()
print(f'Invalid latitude        : {invalid_lat}')
print(f'Invalid longitude       : {invalid_lng}')
coords_per_zip = geo.groupby('geolocation_zip_code_prefix').size()
print(f'Avg coords per ZIP      : {coords_per_zip.mean():.1f}')
print(f'Max coords per ZIP      : {coords_per_zip.max()}')

---
## 10. Data Quality Report

Consolidates all issues found across all 9 tables.

In [ ]:
import os
dq = pd.read_csv('../data/processed/data_quality_report.csv')
dq.style.apply(lambda x: ['background-color: #ffcccc' if v == 'High'
                           else 'background-color: #fff3cc' if v == 'Medium'
                           else '' for v in x], subset=['Severity'])

In [ ]:
print('High severity issues   :', (dq['Severity'] == 'High').sum())
print('Medium severity issues :', (dq['Severity'] == 'Medium').sum())
print('Low severity issues    :', (dq['Severity'] == 'Low').sum())
print()
print('Phase 3 profiling complete.')
print('All findings are documented.')
print('Next: Phase 4 — Cleaning and Staging')